# This will take forever since it is processing file one by one

# Use SHARD

# Module import

In [1]:
# Standard libraries
import json
import os
from pathlib import Path

# Third-party libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from joblib import dump
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# PyTorch core
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader, TensorDataset

# PyTorch Lightning
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

# TorchMetrics
from torchmetrics.classification import (
    BinaryAUROC,
    BinaryAveragePrecision,
    BinaryF1Score,
    BinaryPrecision,
    BinaryRecall
)

# Data preparation

In [3]:
def build_split(input_dir, output_dir):
    all_files = sorted([
        fname for fname in os.listdir(input_dir) if fname.endswith(".pt")
    ])

    train_files, val_files, test_files = [], [], []

    for fname in all_files:
        timestamp = fname.replace("input-", "").replace(".pt", "")
        year = int(timestamp[:4])

        if 2004 <= year <= 2017:
            train_files.append(fname)
        elif 2018 <= year <= 2019:
            val_files.append(fname)
        elif 2020 <= year <= 2024:
            test_files.append(fname)
        else:
            print(f"Skipping unknown year: {year} for file {fname}")

    os.makedirs(output_dir, exist_ok=True)

    pd.Series(train_files).to_csv(os.path.join(output_dir, "train_files.csv"), index=False, header=False)
    pd.Series(val_files).to_csv(os.path.join(output_dir, "val_files.csv"), index=False, header=False)
    pd.Series(test_files).to_csv(os.path.join(output_dir, "test_files.csv"), index=False, header=False)

    print(f"Train: {len(train_files)} files")
    print(f"Val: {len(val_files)} files")
    print(f"Test: {len(test_files)} files")

In [4]:
input_dir = "/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Africa/inputs_t0"
output_dir = "./splits"
build_split(input_dir, output_dir)

Train: 463615 files
Val: 68073 files
Test: 103060 files


In [ ]:
class StormNowcastingDataset(Dataset):
    
    def __init__(self, root_dir, norm_path, file_list, lead_time=0, transform=None):
        """
        root_dir: base folder where inputs_t0 and targets_t{lead_time} exist
        norm_path: path to normalisation.json
        file_list: path to train/val/test csv (one filename per line)
        lead_time: single lead time you are training for (e.g. 0, 1, 2, ...)
        transform: optional PyTorch transforms
        """
        self.input_dir = os.path.join(root_dir, "inputs_t0")
        self.target_dir = os.path.join(root_dir, f"targets_t{lead_time}")
        self.transform = transform

        # Load normalisation parameters
        with open(norm_path, "r") as f:
            self.norm = json.load(f)

        # Load sample filenames
        with open(file_list, "r") as f:
            self.samples = [line.strip() for line in f.readlines()]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        input_filename = self.samples[idx]
        input_path = os.path.join(self.input_dir, input_filename)

        # Match input to target filename by replacing prefix
        timestamp = input_filename.replace("input-", "")
        target_filename = f"target-{timestamp}"
        target_path = os.path.join(self.target_dir, target_filename)

        # Load input and target tensors
        inputs = torch.load(input_path).float()  # shape: (140, 11)
        inputs = self.preprocess_inputs(inputs)  # shape: (140, 10)

        target = torch.load(target_path).float()  # shape: (1024, 1024)

        if self.transform:
            inputs, target = self.transform(inputs, target)

        return inputs, target

    def preprocess_inputs(self, x):
        """
        Apply feature selection, cyclic encoding, and normalisation.

        Original columns: 
        [year, month, day, hour, minute, lat, lon, wp, tir, size, mask]

        Output columns: 
        [month_sin, month_cos, time_sin, time_cos, lat_norm, lon_norm, wp_norm, tir_norm, size_norm, mask]
        """

        # Remove year column (column 0)
        x = x[:, 1:]  # shape becomes (140, 10)

        out = torch.zeros((x.shape[0], 10))

        # Cyclic month encoding
        month = x[:, 0]
        out[:, 0] = torch.sin(2 * np.pi * (month - 1) / 12.0)
        out[:, 1] = torch.cos(2 * np.pi * (month - 1) / 12.0)

        # Cyclic time of day encoding
        hour = x[:, 2]
        minute = x[:, 3]
        time_in_hours = hour + minute / 60.0
        out[:, 2] = torch.sin(2 * np.pi * time_in_hours / 24.0)
        out[:, 3] = torch.cos(2 * np.pi * time_in_hours / 24.0)

        # Latitude normalisation
        lat = x[:, 4]
        out[:, 4] = (lat - self.norm["lat_min"]) / (self.norm["lat_max"] - self.norm["lat_min"])

        # Longitude normalisation
        lon = x[:, 5]
        out[:, 5] = (lon - self.norm["lon_min"]) / (self.norm["lon_max"] - self.norm["lon_min"])

        # Wavelet power: log1p scaling
        wp = x[:, 6]
        out[:, 6] = torch.log1p(wp) / self.norm["wp_max"]

        # TIR normalisation
        tir = x[:, 7]
        out[:, 7] = (tir - self.norm["tir_min"]) / (self.norm["tir_max"] - self.norm["tir_min"])

        # Size normalisation
        size = x[:, 8]
        out[:, 8] = size / self.norm["size_max"]

        # Mask remains unchanged (0 for dummy, 1 for real core)
        out[:, 9] = x[:, 9]

        return out

In [ ]:
root_dir = "/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Africa"
norm_path = "/home/users/mendrika/EPS-Impact-Case-AI-Nowcasting/model/africa/scaling/fict-normalisation.json"
train_file_list = "/home/users/mendrika/EPS-Impact-Case-AI-Nowcasting/model/africa/splits/train_files.csv"

train_dataset = StormNowcastingDataset(
    root_dir=root_dir,
    norm_path=norm_path,
    file_list=train_file_list,
    lead_time=1
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4
)

for batch_inputs, batch_targets in train_loader:
    print(batch_inputs.shape)  # (B, 140, 10)
    print(batch_targets.shape) # (B, 1024, 1024)
    break

In [28]:
class LightningNowcastingModel(pl.LightningModule):
    def __init__(self, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()

        embed_dim = 64

        # Input projection
        self.input_proj = nn.Linear(10, embed_dim)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=256, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # CNN decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, 256, kernel_size=4, stride=4),  # 1x1 -> 4x4
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=4),  # 4x4 -> 16x16
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=4),   # 16x16 -> 64x64
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=4),    # 64x64 -> 256x256
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=4),     # 256x256 -> 1024x1024
            nn.Sigmoid()
        )

    def forward(self, x):
        """
        x: (B, 140, 10)
        """
        B, N, F = x.shape

        x = self.input_proj(x)              # (B, 140, embed_dim)
        x = self.transformer(x)             # (B, 140, embed_dim)
        x = x.mean(dim=1)                   # (B, embed_dim)
        x = x.view(B, -1, 1, 1)             # (B, embed_dim, 1, 1)
        out = self.decoder(x)               # (B, 1, 1024, 1024)
        return out

    def training_step(self, batch, batch_idx):
        inputs, targets = batch
        targets = targets.unsqueeze(1)  # (B, 1, 1024, 1024)
        outputs = self.forward(inputs)

        loss = F.binary_cross_entropy(outputs, targets)
        self.log("train_loss", loss)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        return optimizer

In [ ]:
dataset = StormNowcastingDataset(
    root_dir=root_dir,
    norm_path=norm_path,
    file_list=train_file_list,
    lead_time=3
)

train_loader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=2)

# === Trainer ===

model = LightningNowcastingModel()

trainer = pl.Trainer(
    accelerator="auto",
    devices=1, 
    max_epochs=1  # just for initial test run
)

trainer.fit(model, train_loader)

In [ ]:
test_file_list = "/home/users/mendrika/EPS-Impact-Case-AI-Nowcasting/model/africa/splits/test_files.csv"

# === Load test dataset ===

test_dataset = StormNowcastingDataset(
    root_dir=root_dir,
    norm_path=norm_path,
    file_list=test_file_list,
    lead_time=3
)

test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=2)